# F2 · Obtención, diagnóstico, preparación y validación

**MCDI500 · Grupo 7 · Sumativa 1**

Secuencia: obtener → medir calidad y unidad de observación → decidir tratamientos →
proyectar y transformar → validar → exportar entradas para F3.

**Estado:** base inicial preparada con asistencia de Codex y ejecutada
localmente por Benjamín Araya en Windows. Se completaron 12 celdas de
código y 18 pruebas satisfactorias. Las tres excepciones de conciliación
del detalle permanecen documentadas y pendientes de revisión.



## 1. Obtener y verificar el original

La selección de columnas se realizará con código.

In [1]:
from pathlib import Path
import sys
import json

inicio = Path.cwd().resolve()
ROOT = next((p for p in (inicio, *inicio.parents) if (p / "proyecto.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra Jupyter dentro de proyecto-grupo7-mcdi500.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from F1.src.entorno import cargar_configuracion, verificar_original, reconocer_archivo, registrar_entorno
config = cargar_configuracion(ROOT)
print("Proyecto:", config["proyecto"])
print("Raíz encontrada; las rutas de datos son relativas.")

import pandas as pd
import numpy as np
from F2.src.preprocesamiento import (
    perfil_columnas, proyectar_ordenes, preparar_ordenes, construir_relacion_rubros,
    diagnosticar_detalle, comparar_faltantes, construir_variables, ejecutar_pipeline,
    exportar_resultados, convertir_decimal, marcar_iqr
)
huella_inicial = verificar_original(ROOT, config)
datos = reconocer_archivo(ROOT, config)
print("Lectura:", datos.shape, "| Encoding:", config["datos"]["encoding"])
print("Semilla:", config["semilla"], "(pipeline determinista; no se muestrea)")


Proyecto: Compra Ágil: preparación reproducible de órdenes de compra de Valparaíso
Raíz encontrada; las rutas de datos son relativas.


Lectura: (11177, 63) | Encoding: cp1252
Semilla: 42 (pipeline determinista; no se muestrea)


## 2. Perfilar las 63 columnas

Las proporciones iniciales tienen por denominador las filas exportadas. Después se
vuelven a medir por orden. Contar filas repetidas no autoriza a borrarlas.


In [2]:
perfil = perfil_columnas(datos)
display(perfil)
display(perfil.loc[perfil["faltantes"].gt(0), ["columna", "filas", "faltantes", "porcentaje_faltantes"]])
print("Filas idénticas adicionales:", int(datos.duplicated().sum()))
print("Filas en grupos idénticos:", int(datos.duplicated(keep=False).sum()))


,columna,tipo_lectura,filas,faltantes,porcentaje_faltantes,valores_distintos
0,codigoOC,string,11177,0,0.0,541
1,FechaEnvioOC,string,11177,0,0.0,116
2,NombreOC,string,11177,0,0.0,538
3,DescripcionOC,string,11177,0,0.0,541
4,EstadoOC,string,11177,0,0.0,3
...,...,...,...,...,...,...
58,PrecioTotalItemCotizacion,string,11177,0,0.0,2112
59,MontoNetoCotizacion,string,11177,0,0.0,516
60,MontoDespachoCotizacion,string,11177,0,0.0,24
61,MontoTotalCotizacion,string,11177,0,0.0,519


,columna,filas,faltantes,porcentaje_faltantes
25,ActividadProveedor,11177,2695,24.112016
27,RegionProveedor,11177,150,1.342042
41,ImpuestoEspecificoItem,11177,11177,100.000000


Filas idénticas adicionales: 6


Filas en grupos idénticos: 12


## 3. Unidad de observación y proyección de cabecera

Se comprueba que los 22 atributos seleccionados sean constantes por codigoOC. La
función se detiene ante contradicciones. Solo después proyecta una fila por OC.
No elimina ítems del original ni presupone que las líneas idénticas sean errores.


In [3]:
cabeceras = proyectar_ordenes(datos, config["seleccion"]["columnas_oc"])
display(pd.DataFrame([
    {"unidad": "Fila exportada", "n": len(datos)},
    {"unidad": "OC de cabecera consistente", "n": len(cabeceras)},
    {"unidad": "Proveedor RUT", "n": cabeceras["ProveedorRUT"].nunique()},
]))
print("Selección por código:", config["seleccion"]["columnas_oc"])
for columna in ["MonedaOC", "EstadoOC", "TamanoProveedor"]:
    display(cabeceras[columna].value_counts(dropna=False).rename_axis(columna).to_frame("ordenes"))


,unidad,n
0,Fila exportada,11177
1,OC de cabecera consistente,541
2,Proveedor RUT,277


Selección por código: ['codigoOC', 'FechaEnvioOC', 'EstadoOC', 'ProcedenciaOC', 'MonedaOC', 'MontoNetoOC', 'DescuentosOC', 'CargosOC', 'MontoTotalOC', 'MontoNetoOC_CLP', 'ImpuestosOC_CLP', 'UnidadCompra', 'UnidadCompraRUT', 'RegionUnidadCompra', 'entCode', 'Institucion', 'Sector', 'Proveedor', 'ProveedorRUT', 'ActividadProveedor', 'TamanoProveedor', 'RegionProveedor']


,ordenes
MonedaOC,
CLP,536
UTM,2
USD,2
CLF,1


,ordenes
EstadoOC,
Aceptada,537
Recepcion Conforme,3
Solicitud de Cancelacion,1


,ordenes
TamanoProveedor,
Pequeña,252
Mediana,115
Micro,87
NoClasificado,56
Grande,31


## 4. Conversiones y alcance

Las cifras usan coma decimal; las fechas, día-mes-año. RUT y códigos permanecen como
texto. Se propone el neto CLP informado por la fuente; no se suman totales de monedas
distintas. Se conservan todas las órdenes y sus estados. Una marca propone Aceptada
y Recepción Conforme para análisis descriptivo; el equipo debe aprobar ese filtro.


In [4]:
ordenes, parametros_iqr = preparar_ordenes(cabeceras, config)
display(ordenes.groupby("MonedaOC").agg(ordenes=("codigoOC", "nunique"), neto_clp=("MontoNetoOC_CLP", "sum")))
display(ordenes[["FechaEnvioOC", "MontoNetoOC_CLP", "ProveedorRUT", "mes_envio"]].dtypes.astype(str).to_frame("tipo"))
print("Cobertura:", ordenes["FechaEnvioOC"].min().date(), "a", ordenes["FechaEnvioOC"].max().date())
print("Institución:", cabeceras["Institucion"].unique().tolist())
print("Unidad:", cabeceras["UnidadCompra"].unique().tolist())


,ordenes,neto_clp
MonedaOC,,
CLF,1,139706.289163
CLP,536,661885755.24327
USD,2,1100864.478599
UTM,2,5890589.2


,tipo
FechaEnvioOC,datetime64[us]
MontoNetoOC_CLP,Float64
ProveedorRUT,string
mes_envio,string


Cobertura: 2025-01-06 a 2025-06-30
Institución: ['I MUNICIPALIDAD DE VALPARAISO']
Unidad: ['ABASTECIMIENTO']


## 5. Faltantes: alternativas e impacto

Eliminar órdenes por actividad o región ausente reduce cobertura. Imputar la moda
atribuye información que la fuente no aporta. Se conservan los NA y se agrega una
marca y una etiqueta de presentación separada. ImpuestoEspecificoItem, totalmente
vacío, no se interpreta como cero ni se incorpora a la cabecera.

El neto CLP no tiene faltantes; no procede imputar por media, mediana ni grupo.
La varianza antes/después es igual porque ningún monto se imputa ni recorta.


In [5]:
comparacion = comparar_faltantes(cabeceras, ["ActividadProveedor", "RegionProveedor"])
display(comparacion)
antes = convertir_decimal(cabeceras["MontoNetoOC_CLP"]).to_numpy(dtype=float)
despues = ordenes["MontoNetoOC_CLP"].to_numpy(dtype=float)
display(pd.DataFrame([
    {"etapa": "Antes", "n": len(antes), "faltantes": int(np.isnan(antes).sum()), "varianza_ddof0": np.var(antes)},
    {"etapa": "Después, sin imputación monetaria", "n": len(despues), "faltantes": int(np.isnan(despues).sum()), "varianza_ddof0": np.var(despues)},
]))
np.testing.assert_allclose(antes, despues, rtol=0, atol=0)


,columna,estrategia,ordenes_iniciales,faltantes_originales,ordenes_retenidas,atributos_reales_asignados_sin_evidencia
0,ActividadProveedor,eliminar_orden_sin_dato,541,139,402,0
1,ActividadProveedor,imputar_moda,541,139,541,139
2,ActividadProveedor,conservar_NA_y_marca,541,139,541,0
3,RegionProveedor,eliminar_orden_sin_dato,541,8,533,0
4,RegionProveedor,imputar_moda,541,8,541,8
5,RegionProveedor,conservar_NA_y_marca,541,8,541,0


,etapa,n,faltantes,varianza_ddof0
0,Antes,541,0,2.326586e+12
1,"Después, sin imputación monetaria",541,0,2.326586e+12


## 6. Proveedores y rubros

Se normalizan espacios, mayúsculas y puntos del RUT para comparar, sin fusiones
difusas ni validación fiscal. Se cuenta cuántos RUT presentan varios nombres.

N1/N2/N3 se conservan como ruta jerárquica. La tabla puente expresa pertenencia;
no tiene montos. Una OC puede participar en varios rubros: sus conteos se solapan.


In [6]:
nombres = ordenes.groupby("ProveedorRUT_normalizado")["Proveedor_nombre_comparable"].nunique()
print("RUT con varios nombres normalizados:", int(nombres.gt(1).sum()))
relacion = construir_relacion_rubros(datos)
display(pd.DataFrame({"nivel": ["RubroN1", "RubroN2", "RubroN3"], "categorias": [relacion[c].nunique() for c in ["RubroN1", "RubroN2", "RubroN3"]]}))
print("OC con varios N1:", int(relacion.groupby("codigoOC")["RubroN1"].nunique().gt(1).sum()))
print("Etiquetas N2 con varios padres N1:", int(relacion.groupby("RubroN2")["RubroN1"].nunique().gt(1).sum()))
display(relacion[["RubroN1", "RubroN2", "RubroN3"]].drop_duplicates().head())


RUT con varios nombres normalizados: 0


,nivel,categorias
0,RubroN1,53
1,RubroN2,191
2,RubroN3,487


OC con varios N1: 180
Etiquetas N2 con varios padres N1: 1


,RubroN1,RubroN2,RubroN3
0,"Servicios de Viajes, alimentación, alojamiento...",Restaurantes y catering,Servicios de comedor y banquetes
2,"Equipos, suministros y accesorios deportivos y...",Coleccionismo y condecoraciones o premios,Premios
3,Artículos de fabricación y producción,Artículos de ferretería,Tornillos
4,Artículos de fabricación y producción,Cintas adhesivas y selladores,Adhesivos y selladores
5,Artículos de fabricación y producción,Cintas adhesivas y selladores,Cinta adhesiva


## 7. Auditar detalle y repeticiones

Se comparan firmas distintas de ítem y cotización. La coincidencia entre filas y
producto de firmas es compatible con una expansión cartesiana; no demuestra por
sí sola el mecanismo de exportación.

La conciliación diagnóstica suma MontoNetoItemCLP de firmas distintas y compara con
MontoNetoOC_CLP, tolerancia absoluta de 1 CLP. Conciliar no prueba unicidad real de
líneas. Tres discrepancias impiden presentar esas firmas como detalle definitivo.


In [7]:
conciliacion, cruce = diagnosticar_detalle(datos, ordenes, config)
print("Grupos OC-producto:", len(cruce))
print("Compatibles con producto de firmas:", int(cruce["coincide_producto"].sum()))
display(cruce.loc[cruce["codigoOC"].eq("2427-699-AG25")])
excepciones = conciliacion.loc[~conciliacion["concilia_firmas_a_1_clp"]]
display(excepciones)
print("OC con conciliación numérica de firmas:", int(conciliacion["concilia_firmas_a_1_clp"].sum()))
display(datos.loc[datos["codigoOC"].eq("2427-264-AG25"), ["codigoOC", "CantidadItem", "MontoNetoItemCLP", "MontoNetoOC_CLP"]])
print("En ese caso, borrar una fila idéntica quitaría 1.000.000 CLP del detalle.")


Grupos OC-producto: 2043
Compatibles con producto de firmas: 2040


,codigoOC,CodigoProductoONU,filas_exportadas,firmas_item,firmas_cotizacion,producto_firmas,coincide_producto
1958,2427-699-AG25,14111509,1024,32,32,1024,True


,codigoOC,MontoNetoOC_CLP,neto_linea_clp_diagnostico,diferencia_clp,concilia_firmas_a_1_clp
137,2427-264-AG25,2000000.0,1000000.0,-1000000.0,False
346,2427-511-AG25,37473.0,29984.0,-7489.0,False
380,2427-543-AG25,1490341.0,1463816.0,-26525.0,False


OC con conciliación numérica de firmas: 538


,codigoOC,CantidadItem,MontoNetoItemCLP,MontoNetoOC_CLP
3075,2427-264-AG25,1,1000000,2000000
3076,2427-264-AG25,1,1000000,2000000


En ese caso, borrar una fila idéntica quitaría 1.000.000 CLP del detalle.


## 8. Extremos: detectar y conservar

Se aplica Q3 + 1,5·IQR al neto CLP de todas las 541 OC y, por separado, se reproduce
el diagnóstico del mapa sobre MontoTotalOC de las 536 OC cuya moneda es CLP.
Cambian variable y población, por lo que los conteos pueden diferir.


In [8]:
solo_clp = ordenes.loc[ordenes["MonedaOC"].eq("CLP"), "MontoTotalOC"]
marcas_mapa, parametros_mapa = marcar_iqr(solo_clp)
display(pd.DataFrame([
    {"base": "Neto CLP, todas las OC", **parametros_iqr, "extremos": int(ordenes["extremo_iqr_neto_clp"].sum())},
    {"base": "Total OC, solo CLP; mapa", **parametros_mapa, "extremos": int(marcas_mapa.sum())},
]))
print("Montos eliminados o recortados: 0.")


,base,n,q1,q3,limite_inferior,limite_superior,factor,extremos
0,"Neto CLP, todas las OC",541,217923.0000,1620152.00,-1.885420e+06,3.723496e+06,1.5,61
1,"Total OC, solo CLP; mapa",536,258756.2775,1906960.72,-2.213550e+06,4.379267e+06,1.5,60


Montos eliminados o recortados: 0.


## 9. Transformaciones y parámetros

Una vista separada contiene Z del neto CLP, one-hot de moneda/estado y tamaño ordinal
Micro < Pequeña < Mediana < Grande. NoClasificado queda NA con indicador. La
clasificación procede del archivo; no se verificó con ventas del proveedor.

La vista apoya futuras exploraciones, no constituye un modelo entrenado. Si se
modela después, parámetros y vocabularios deben ajustarse solo con entrenamiento.
No se usa Z para sumar dinero ni el ordinal para suponer distancias iguales.


In [9]:
variables, parametros = construir_variables(ordenes, config)
print("Vista transformada local:", variables.shape, "| Columnas:", variables.columns.tolist())
print(json.dumps(parametros, ensure_ascii=False, indent=2))
print("Tamaños sin clasificación:", int(variables["tamano_sin_clasificar"].sum()))
assert len(variables) == len(ordenes)


Vista transformada local: (541, 11) | Columnas: ['codigoOC', 'neto_clp_z', 'tamano_ordinal', 'tamano_sin_clasificar', 'MonedaOC_CLF', 'MonedaOC_CLP', 'MonedaOC_USD', 'MonedaOC_UTM', 'EstadoOC_Aceptada', 'EstadoOC_Recepcion Conforme', 'EstadoOC_Solicitud de Cancelacion']
{
  "estandarizacion": {
    "media": 1236630.1575065302,
    "desviacion": 1525315.0851613507,
    "ddof": 0
  },
  "orden_tamano": [
    "Micro",
    "Pequeña",
    "Mediana",
    "Grande"
  ],
  "columnas_nominales": [
    "MonedaOC_CLF",
    "MonedaOC_CLP",
    "MonedaOC_USD",
    "MonedaOC_UTM",
    "EstadoOC_Aceptada",
    "EstadoOC_Recepcion Conforme",
    "EstadoOC_Solicitud de Cancelacion"
  ]
}
Tamaños sin clasificación: 56


## 10. Validar casos normales, límites y errores

Las pruebas revisan formatos, fechas imposibles, montos contradictorios, conservación
de datos, RUT con ceros iniciales, series constantes, categorías inesperadas, una
sola categoría y las tres excepciones reales. Un fallo imprevisto detiene el notebook.


In [10]:
import unittest
import io
suite = unittest.defaultTestLoader.discover(str(ROOT / "F2/tests"), pattern="test_*.py")
salida = io.StringIO()
pruebas = unittest.TextTestRunner(stream=salida, verbosity=2).run(suite)
print(salida.getvalue())
assert pruebas.wasSuccessful(), "Hay pruebas fallidas."
resumen = {"pruebas": pruebas.testsRun, "fallos": len(pruebas.failures), "errores": len(pruebas.errors), "exitosas": pruebas.wasSuccessful()}
(ROOT / "evidencias").mkdir(exist_ok=True)
(ROOT / "evidencias/pruebas_F2.json").write_text(json.dumps(resumen, indent=2) + "\n", encoding="utf-8")


test_no_ocultar_tres_excepciones_de_detalle (test_preprocesamiento.TestArchivoReal.test_no_ocultar_tres_excepciones_de_detalle) ... ok
test_preservar_montos_estados_y_faltantes (test_preprocesamiento.TestArchivoReal.test_preservar_montos_estados_y_faltantes) ... ok
test_proyeccion_y_claves_foraneas (test_preprocesamiento.TestArchivoReal.test_proyeccion_y_claves_foraneas) ... ok
test_transformaciones_y_denominador_del_mapa (test_preprocesamiento.TestArchivoReal.test_transformaciones_y_denominador_del_mapa) ... ok
test_columna_faltante_y_codigo_vacio (test_preprocesamiento.TestFunciones.test_columna_faltante_y_codigo_vacio) ... ok
test_coma_decimal_miles_y_nulo (test_preprocesamiento.TestFunciones.test_coma_decimal_miles_y_nulo) ... ok
test_decimal_entrada_vacia (test_preprocesamiento.TestFunciones.test_decimal_entrada_vacia) ... ok
test_estandarizacion_constante_y_todo_nulo (test_preprocesamiento.TestFunciones.test_estandarizacion_constante_y_todo_nulo) ... ok
test_faltantes_sin_nulos_n

71

## 11. Exportar, recargar y verificar

Se vuelve a ejecutar el pipeline como función para no depender de variables editadas
manualmente. La recarga preserva códigos y comprueba claves/montos con tolerancia
acorde con los ocho decimales del CSV exportado.


In [11]:
resultado = ejecutar_pipeline(ROOT)
exportar_resultados(resultado, ROOT)
recarga = pd.read_csv(ROOT / "F2/data/processed/ordenes.csv", dtype={"codigoOC": "string", "ProveedorRUT": "string", "ProveedorRUT_normalizado": "string"})
assert len(recarga) == len(resultado["ordenes"]) and recarga["codigoOC"].is_unique
assert recarga["codigoOC"].tolist() == resultado["ordenes"]["codigoOC"].tolist()
np.testing.assert_allclose(recarga["MontoNetoOC_CLP"], resultado["ordenes"]["MontoNetoOC_CLP"].to_numpy(dtype=float), rtol=0, atol=1e-7)
assert verificar_original(ROOT, config) == huella_inicial
print("Exportación y recarga verificadas; original intacto.")
display(pd.DataFrame([{"tabla": k, "filas": len(resultado[k]), "columnas": len(resultado[k].columns)} for k in ["ordenes", "relacion_rubros", "variables", "excepciones_detalle"]]))


Exportación y recarga verificadas; original intacto.


,tabla,filas,columnas
0,ordenes,541,31
1,relacion_rubros,1683,4
2,variables,541,11
3,excepciones_detalle,3,5


## 12. Alimentar el informe

La figura y las tablas LaTeX se generan desde las mismas métricas.

In [12]:
from F2.src.informe import generar_insumos_informe
generar_insumos_informe(resultado, ROOT)
entorno = registrar_entorno()
(ROOT / "evidencias/entorno_F2.json").write_text(json.dumps(entorno, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Insumos generados en informe/tablas e informe/figuras.")
print("Compilar desde Git Bash: python scripts/compilar_informe.py")


Insumos generados en informe/tablas e informe/figuras.
Compilar desde Git Bash: python scripts/compilar_informe.py


## 13. Reflexión y pendientes

La dificultad principal es la mezcla de unidades en el archivo. Validar atributos
constantes permite formar cabeceras sin multiplicar montos. Pero borrar firmas
idénticas de detalle deja tres discrepancias: se conserva la evidencia y se aplaza
el reparto de montos por rubro.

Las ausencias se conservan porque imputar moda inventaría actividades o regiones.
Los extremos permanecen porque un criterio estadístico no demuestra error.
El cambio del conteo de extremos frente al mapa tiene explicación comprobable:
la variable y el denominador son distintos.

**Pendiente del equipo:** diccionario monetario, multiplicidades del detalle, filtro
de estados, finalidad de transformaciones, URL/fecha de descarga, ejecución en los demás equipos,
reconstrucción de dependencias, revisión por pares, portada y aportes reales.

Consultar docs/vinculacion_mapa.md y docs/bitacora_decisiones.md.
